# Unconstrained section — every figure and number

Regenerates the whole unconstrained results section from `results/auto/`.
Logic lives in `paper_figures.py` so it can also be run headless
(`python paper_figures.py`) and so these cells stay readable.

**Three rules this encodes** (decisions from calls, not taste):

1. **Seeds are never pooled.** A `(reward, prompt, seed)` triple is one
   optimization problem — different `x0`, different uncontrolled image,
   sometimes a different subject. Pooling was rejected on the 2026-09-17 call.
2. **Plots come from `sweep_diagnostics.py` / `compare_methods.py`**, not a
   reimplementation, so the encoding matches what Shervin already reviews.
3. **Win rates are computed from `summary.csv`**, never by summing
   `winrates.csv` — there are two generations of those on disk (pooled-seed
   under `diagnostics/`, per-seed under `diag_seed*/`) and adding them
   double-counts.

Needs `ipykernel` in the venv: `uv add --dev ipykernel` (or just run the `.py`).

In [ ]:
import paper_figures as pf

df = pf.load_rows()
print(f"{len(df)} successful rows, {df.groupby(['reward','prompt','seed']).ngroups} combos")
df.groupby(['reward', 'prompt', 'seed']).size().rename('rows').to_frame()

## Numerical result: win rates

Per `(reward, prompt, seed, lam)` cell, does MPC's best config beat each
baseline on `objective` (minimised)?

Report **both** the unfiltered number and the energy-capped ones. Some of MPC's
margin comes from HardFlow/FMRG *diverging* at high lambda rather than from MPC
being better, and the cap shows how much. It is the first thing a reviewer will
poke at, so pre-empt it in the text.

In [ ]:
pf.win_rate_summary(df)

In [ ]:
# Per combo. The pooled number is unevenly weighted -- the densified combos
# contribute ~11 cells each vs 6 -- so quote this table alongside it.
pf.win_rate_by_combo(df)

## Which combo to put in the main text

`mpc_share` = fraction of that seed's globally non-dominated set owned by MPC.
Combos missing any of the three methods are dropped, not ranked: an absent
baseline hands MPC 100% for free.

In [ ]:
pf.frontier_share(df)

## The headline table

Cheapest control energy each method needs to *reach* a reward level. This is the
claim the Pareto plot makes, in numbers.

In [ ]:
pf.energy_at_reward(df, 'pickscore', 'a portrait of a man', '42',
                    [21, 23, 25, 28, 30])

In [ ]:
# Replication on an independent seed -- this is what makes it a result
# rather than an anecdote.
pf.energy_at_reward(df, 'pickscore', 'a portrait of a man', '7',
                    [21, 23, 25, 28, 30])

## Regenerate every figure

Per-seed Pareto into `diag_seed<N>/` and matched-operating-point image grids
into `compare_seed<N>/`, then flattened into `results/paper_figures/` with
self-describing filenames ready to drop into Overleaf.

Takes a couple of minutes and is CPU-only — it reads saved PNGs and CSVs, so it
is safe to run while a sweep has the GPUs.

In [ ]:
for line in pf.make_per_seed_figures():
    print(line)

In [ ]:
figs = pf.collect_paper_figures()
print(f"{len(figs)} figures -> {pf.FIGS}")
for f in figs:
    print(' ', f.name)

## What is actually in the draft

Five figures, tracked in `pf.DRAFT_FIGURES` in reading order. The Overleaf
filename is not always the generated one — the body Pareto was uploaded with an
`unconstrained_` prefix — so the mapping is recorded explicitly and
`pf.export_draft_figures()` copies under the Overleaf name.

| where | reward / prompt / seed | Overleaf filename |
|---|---|---|
| body, Pareto | PickScore · portrait · 42 | `unconstrained_pareto__pickscore__…__seed42.png` |
| body, image rows (a) | PickScore · portrait · 42 | `images_reward__pickscore__…__seed42.png` |
| body, image rows (b) | PickScore · portrait · 7 | `images_reward__pickscore__…__seed7.png` |
| appendix, Pareto (a) | ImageReward · sleeping cat · 42 | `pareto__imagereward__a_photo_of___seed42.png` |
| appendix, Pareto (b) | CLIP · portrait · 7 | `pareto__clip__…__seed7.png` |

The two appendix panels were chosen to vary reward, prompt pair **and** seed
relative to the body figure, so between them the three figures cover three
rewards, two prompt pairs and two seeds.

**Not appendix material:** `/home/sungcho/figures/fig2_pareto_*.png`. Those came
from `fig2_pareto_tuning.ipynb` as a λ-tuning artifact off the older Penn-box
lam-suite runs, cover only `a portrait of a man`, and aggregate across seeds as
`J med / r med / E med` — seed pooling is exactly what the 09-17 call rejected.

**Citations for the two task definitions:**

- image editing — HardFlow (arXiv 2511.08425), five face edits on CelebA-HQ with
  an LPIPS identity constraint
- aesthetic enhancement — FMRG (arXiv 2604.27147) Fig. 12, and ReNO
  (arXiv 2406.04312) whose reward ensemble FMRG adopts

**Known gap.** The enhancement task has no three-method data yet — only an
FMRG-at-N=4 prompt screen (`results/screen2/`) used to choose prompts. That N
does not match the editing runs' N=8, H=2, so it cannot be reused as the FMRG
arm. Until that sweep runs there is no enhancement Pareto and no enhancement
win rate, and the draft's image rows are editing-only.

**Baseline caveat to resolve before publishing these numbers.** FMRG's own
config for reward-guided generation uses early stopping at `t_stop=0.25` and
K=3 warmup particles; their paper states removing early stopping causes
over-optimization artifacts. Our FMRG arm has neither, which is the likely cause
of the diverged FMRG energies visible in the high-lambda cells. Note the 09-20
"let's keep it as is" decision was about HardFlow's 0.02 margin, *not* this.

In [ ]:
# Does every figure the draft references exist on disk?
pf.draft_manifest()

In [ ]:
# Every draft figure inline, in reading order, with the per-instance numbers
# its caption asserts underneath.
from IPython.display import Image, display, Markdown

for f in pf.DRAFT_FIGURES:
    s = pf.combo_stats(df, *f['combo'])
    display(Markdown(
        f"**{f['role']}** — {pf.PRETTY_REWARD[s['reward']]}, "
        f"\"{s['prompt']}\" → \"{s['edit_prompt']}\", seed {s['seed']}  \n"
        f"`{f['overleaf']}`  \n"
        f"{s['cells']} cells · beats HardFlow {s['wins_hardflow']}/{s['cells']} · "
        f"beats FMRG {s['wins_fmrg']}/{s['cells']} · "
        f"owns {s['frontier_mpc']}/{s['frontier_n']} of the frontier · "
        f"{s['n_lam']} λ in [{s['lam_min']:g}, {s['lam_max']:g}]"))
    display(Image(str(pf.FIGS / f['generated'])))

## Export just the draft figures

`results/paper_figures/` holds all 36 generated figures; the draft uses five of
them. This writes those five into `results/draft_figures/` under their Overleaf
names, so re-uploading that one directory is the whole sync step.

It raises on a missing source instead of exporting a partial set — a draft
quietly missing one figure still compiles, and shows the wrong thing.

In [ ]:
for p in pf.export_draft_figures():
    print(p.name)
print(f'\n-> {pf.DRAFT}')

## The LaTeX blocks

The three figure environments in the draft, emitted with every quoted number
pulled from `summary.csv` via `pf.combo_stats` — frontier counts, cells, wins
against each baseline, N, H, and the edit prompt text. Same reason win rates are
not summed from `winrates.csv`: a number typed into a caption cannot be checked
against the figure beside it, and these move whenever a baseline config changes.

Individually: `pf.latex_body_pareto(df)`, `pf.latex_body_image_rows(df)`,
`pf.latex_appendix_pareto(df)`. All take `prefix=` if the figures do not live in
`figures/` in the Overleaf project.

Both `figure*` blocks need `\usepackage{subcaption}` in the preamble — and not
`subfigure` or `subfig` alongside it, which conflict.

In [ ]:
# Paste into the draft. Every quoted number is interpolated from combo_stats,
# so re-run this after any sweep that adds cells and re-paste rather than
# hand-editing a caption in the .tex.
print(pf.latex_all(df))